# Can we predict a warranty claim within a year?
Target: `claim_within_year` (~12.6% positives). Follows `checklist.md`.

**Data notes (from `columns.csv` and inspection):**
- `device_serial` is an identifier -> dropped. It is unique per row, so no entity grouping is needed.
- There is no date/time column, so a "latest rows" split is impossible; row order is not assumed to be time. We use a stratified random split. If the model is meant for future sales, drift is untested.
- Remaining features (channel, device age at sale, price, reported daily use at registration) are known at sale/registration, before the outcome.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
df = pd.read_csv('data.csv')
print(df.shape, df.device_serial.is_unique, df.isna().sum().sum())
df = df.drop(columns='device_serial')
print(df.describe(include='all'))

## 1. Set aside the test set first (20%, stratified, untouched until step 6)

In [ ]:
X, y = df.drop(columns='claim_within_year'), df.claim_within_year
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
print(len(X_tr), len(X_te), y_tr.mean(), y_te.mean())

## 2-4. Candidates (named before any scoring)
1. `baseline`: predicts the training prior (trivial)
2. `logreg`: logistic regression, C in {0.1, 1, 10}
3. `hgb`: gradient boosting, max_depth in {2, 3}, learning_rate 0.05, 150 iterations

Metrics: ROC AUC and average precision; threshold chosen on out-of-fold predictions by F1. Preprocessing is inside the pipeline so it is refit in each CV fold.

In [ ]:
num = ['device_age_years','price_eur','daily_hours']; cat = ['channel']
def pre(scale):
    return ColumnTransformer([('n', StandardScaler() if scale else 'passthrough', num),
                              ('c', OneHotEncoder(handle_unknown='ignore'), cat)])
cands = {'baseline': Pipeline([('p',pre(False)),('m',DummyClassifier(strategy='prior'))])}
for C_ in [0.1,1,10]:
    cands[f'logreg_C{C_}'] = Pipeline([('p',pre(True)),('m',LogisticRegression(C=C_,max_iter=1000))])
for d in [2,3]:
    cands[f'hgb_d{d}'] = Pipeline([('p',pre(False)),('m',HistGradientBoostingClassifier(max_depth=d,learning_rate=0.05,max_iter=150,random_state=0))])
print(list(cands))

## 2-3. Cross-validation on training rows only (stratified 5-fold x 3 repeats; no repeated entities, so plain folds)

In [ ]:
rows=[]; oof={}
for name,m in cands.items():
    aucs=[];aps=[];ps=[]
    for r in range(3):
        cv=StratifiedKFold(5,shuffle=True,random_state=r)
        p=cross_val_predict(m,X_tr,y_tr,cv=cv,method='predict_proba')[:,1]
        aucs.append(roc_auc_score(y_tr,p)); aps.append(average_precision_score(y_tr,p)); ps.append(p)
    oof[name]=np.mean(ps,axis=0)
    rows.append((name,np.mean(aucs),np.std(aucs),np.mean(aps)))
cvres=pd.DataFrame(rows,columns=['model','cv_auc','auc_sd_over_repeats','cv_ap']).sort_values('cv_auc',ascending=False)
print(cvres)

## 5. Choose model and threshold on CV only

In [ ]:
best = cvres[cvres.model!='baseline'].iloc[0].model
p = oof[best]
ths = np.linspace(0.02,0.6,59)
f1s = [f1_score(y_tr,p>=t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
r = cvres.set_index('model')
print('chosen model:', best, '| CV AUC', r.loc[best,'cv_auc'].round(3), '| baseline CV AUC', r.loc['baseline','cv_auc'].round(3))
print('chosen threshold (max OOF F1):', round(thr,3), 'OOF F1', round(max(f1s),3))

**Choice recorded before test scoring:** model and threshold printed above (best CV AUC among non-baseline candidates; threshold maximising out-of-fold F1). Nothing below changes them.

## 6. Refit on all training rows; score the test set once

In [ ]:
final = cands[best].fit(X_tr,y_tr)
pt = final.predict_proba(X_te)[:,1]
base = cands['baseline'].fit(X_tr,y_tr).predict_proba(X_te)[:,1]
print(f'TEST AUC {roc_auc_score(y_te,pt):.3f} | AP {average_precision_score(y_te,pt):.3f} (prevalence {y_te.mean():.3f}) | F1@thr {f1_score(y_te,pt>=thr):.3f}')
print(f'baseline: AUC {roc_auc_score(y_te,base):.3f} AP {average_precision_score(y_te,base):.3f} F1 {f1_score(y_te,base>=thr):.3f}')
rng=np.random.default_rng(0); yt=y_te.values
bs=[roc_auc_score(yt[i],pt[i]) for i in (rng.integers(0,len(yt),len(yt)) for _ in range(500))]
print('AUC 95% CI', np.percentile(bs,[2.5,97.5]).round(3))

Caveats: no time column so the split is random; one test set of 1,400 rows (~176 positives); `daily_hours` has implausible values >24, left as-is.